In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import torch.nn as nn

import torchvision.transforms as T
import matplotlib.pyplot as plt


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_PATH = "../../results/_gen_stage19_combo_FID/generated/scene_0001_combo_smooth2x.png"
IMAGE_SIZE = 256
IMAGE_MODE = "rgb"

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

if IMAGE_MODE == "rgb":
    transform = T.Compose([
        T.Resize(int(IMAGE_SIZE * 1.10)),
        T.CenterCrop(IMAGE_SIZE),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
else:
    transform = T.Compose([
        T.Resize(int(IMAGE_SIZE * 1.10)),
        T.CenterCrop(IMAGE_SIZE),
        T.ToTensor(),
    ])


# --------------------------------
# EDIT THIS LIST ONLY
# --------------------------------
MODEL_SPECS = [
    {
        "name": "Stage1",
        "ckpt": "../../CNN_models/classifier/Stage1/Stage1.pt",
        "build": lambda: SimpleCNN_GAP1(num_classes=2),
        "class_names": {0: "Non-Empty", 1: "Empty"},
    },
    {
        "name": "Stage2",
        "ckpt": "../../CNN_models/classifier/Stage2/Stage2.pt",
        "build": lambda: SimpleCNN_GAP2(num_classes=2),
        "class_names": {0: "Overlap", 1: "Isolated"},
    },
    {
        "name": "Stage3",
        "ckpt": "../../CNN_models/classifier/Stage3/Stage3.pt",
        "build": lambda: SimpleCNN_GAP2(num_classes=2),
        "class_names": {0: "Cluttered", 1: "Not-cluttered"},
    },
]


class SimpleCNN_GAP1(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(64, 64), nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


class SimpleCNN_GAP2(nn.Module):
    """Custom CNN with Global Average Pooling"""
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(512, 512), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


def load_model(build_fn, ckpt_path: str):
    model = build_fn().to(DEVICE)

    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt:
            state = ckpt["model_state_dict"]
        elif "model_state" in ckpt:
            state = ckpt["model_state"]
        else:
            state = ckpt
    else:
        state = ckpt

    if any(k.startswith("module.") for k in state.keys()):
        state = {k.replace("module.", "", 1): v for k, v in state.items()}

    model.load_state_dict(state, strict=True)
    model.eval()
    return model


@torch.no_grad()
def predict_one(model, x, class_names: dict[int, str]):
    logits = model(x)
    probs = torch.softmax(logits, dim=1)

    pred_id = int(probs.argmax(dim=1).item())
    confidence = float(probs.max(dim=1).values.item())
    pred_name = class_names[pred_id]

    return pred_id, pred_name, confidence, probs.squeeze(0).cpu().tolist()


def format_class_mapping(class_names: dict[int, str]) -> str:
    parts = [f"{idx}={name}" for idx, name in sorted(class_names.items())]
    return ", ".join(parts)


def main():
    img = Image.open(IMAGE_PATH).convert("RGB")

    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Input image")
    plt.show()

    x = transform(img).unsqueeze(0).to(DEVICE)

    print("\n===== PREDICTIONS (1 image → 3 models) =====")
    for spec in MODEL_SPECS:
        model = load_model(spec["build"], spec["ckpt"])
        pred_id, pred_name, conf, probs = predict_one(model, x, spec["class_names"])

        print(f"\n{spec['name']}")
        print(f"  Class meaning: {format_class_mapping(spec['class_names'])}")
        print(f"  Predicted: {pred_name} (class {pred_id})")
        print(f"  Confidence: {conf:.4f}")

        print("  Per-class probabilities:")
        for class_id, class_name in sorted(spec["class_names"].items()):
            print(f"    class {class_id} = {class_name}: {probs[class_id]:.4f}")

    print("\n===========================================\n")


if __name__ == "__main__":
    main()